In [1]:
import re
import json
import time
import hashlib
import random
import zipfile
from pathlib import Path
from urllib.parse import urljoin, parse_qs, urlparse, unquote

import requests
import urllib3
import pandas as pd
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


BASE_URL = "https://zakupki.gov.ru"
SEARCH_URL = BASE_URL + "/epz/order/extendedsearch/results.html"

CUSTOMER_INN = "5008006211"  
SEARCH_STRINGS = [
    '"Московский физико-технический институт"',
]

SEARCH_URL_FROM_BROWSER = "https://zakupki.gov.ru/epz/order/extendedsearch/results.html?searchString=%D0%9C%D0%BE%D1%81%D0%BA%D0%BE%D0%B2%D1%81%D0%BA%D0%B8%D0%B9+%D0%A4%D0%B8%D0%B7%D0%B8%D0%BA%D0%BE-%D1%82%D0%B5%D1%85%D0%BD%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B9+%D0%B8%D0%BD%D1%81%D1%82%D0%B8%D1%82%D1%83%D1%82&morphology=on&sortBy=UPDATE_DATE&pageNumber=1&sortDirection=false&recordsPerPage=_10&showLotsInfoHidden=false&fz223=on&af=on&ca=on&pc=on&pa=on&priceContractAdvantages44IdNameHidden=%7B%7D&currencyIdGeneral=-1&customerIdOrg=10481000002%3A%D0%A4%D0%95%D0%94%D0%95%D0%A0%D0%90%D0%9B%D0%AC%D0%9D%D0%9E%D0%95+%D0%93%D0%9E%D0%A1%D0%A3%D0%94%D0%90%D0%A0%D0%A1%D0%A2%D0%92%D0%95%D0%9D%D0%9D%D0%9E%D0%95+%D0%90%D0%92%D0%A2%D0%9E%D0%9D%D0%9E%D0%9C%D0%9D%D0%9E%D0%95+%D0%9E%D0%91%D0%A0%D0%90%D0%97%D0%9E%D0%92%D0%90%D0%A2%D0%95%D0%9B%D0%AC%D0%9D%D0%9E%D0%95+%D0%A3%D0%A7%D0%A0%D0%95%D0%96%D0%94%D0%95%D0%9D%D0%98%D0%95+%D0%92%D0%AB%D0%A1%D0%A8%D0%95%D0%93%D0%9E+%D0%9E%D0%91%D0%A0%D0%90%D0%97%D0%9E%D0%92%D0%90%D0%9D%D0%98%D0%AF+%22%D0%9C%D0%9E%D0%A1%D0%9A%D0%9E%D0%92%D0%A1%D0%9A%D0%98%D0%99+%D0%A4%D0%98%D0%97%D0%98%D0%9A%D0%9E-%D0%A2%D0%95%D0%A5%D0%9D%D0%98%D0%A7%D0%95%D0%A1%D0%9A%D0%98%D0%99+%D0%98%D0%9D%D0%A1%D0%A2%D0%98%D0%A2%D0%A3%D0%A2+%28%D0%9D%D0%90%D0%A6%D0%98%D0%9E%D0%9D%D0%90%D0%9B%D0%AC%D0%9D%D0%AB%D0%99+%D0%98%D0%A1%D0%A1%D0%9B%D0%95%D0%94%D0%9E%D0%92%D0%90%D0%A2%D0%95%D0%9B%D0%AC%D0%A1%D0%9A%D0%98%D0%99+%D0%A3%D0%9D%D0%98%D0%92%D0%95%D0%A0%D0%A1%D0%98%D0%A2%D0%95%D0%A2%29%22zZ10481000002zZ10481000002zZzZ5008006211zZ-1zZ772701001zZ1027739386135&selectedSubjectsIdNameHidden=%7B%7D&koksIdsIdNameHidden=%7B%7D&gws=%D0%92%D1%8B%D0%B1%D0%B5%D1%80%D0%B8%D1%82%D0%B5+%D1%82%D0%B8%D0%BF+%D0%B7%D0%B0%D0%BA%D1%83%D0%BF%D0%BA%D0%B8"
TRUST_URL_FILTER = bool(SEARCH_URL_FROM_BROWSER)
CUSTOMER_RE = re.compile(r"московск\w*\s+физико-технич\w*\s+институт|" + CUSTOMER_INN, re.I)

USE_44FZ = False        
USE_223FZ = True        

MAX_PURCHASES = None     
MAX_PER_RUN = None       
MAX_EMPTY_PAGES = 3     
MAX_PAGES = 100          
PAGE_SIZE = 50           

DOWNLOAD_ONLY_TZ = True  
UNPACK_ZIP = True        

PAUSE_MIN, PAUSE_MAX = 2.0, 4.5   
LONG_PAUSE_EVERY = 100            
LONG_PAUSE = (2, 4)           
MAX_CONSECUTIVE_FAILURES = 5      
VERIFY_SSL = False

OUT_DIR = Path("mipt_tz_dataset")
RAW_DIR = OUT_DIR / "raw"
DEBUG_DIR = OUT_DIR / "debug"
INDEX_FILE = OUT_DIR / "documents_index.jsonl"

for d in (RAW_DIR, DEBUG_DIR):
    d.mkdir(parents=True, exist_ok=True)


TZ_RE = re.compile(
    r"техническ|(?<![а-яё])тз(?![а-яё])|(?<![а-яё])т\.\s?з\.?(?![а-яё])|техзадан|задани"
    r"|описани\w*\s+объект|спецификац|документац|извещени|приложени",
    re.I,
)

EXCLUDE_RE = re.compile(r"протокол|разъяснен", re.I)


def is_tz_doc(name):
    return bool(TZ_RE.search(name)) and not EXCLUDE_RE.search(name)


FILTER_SIG = hashlib.md5(
    (TZ_RE.pattern + EXCLUDE_RE.pattern + str(DOWNLOAD_ONLY_TZ)).encode()
).hexdigest()[:8]


session = requests.Session()
session.headers.update({
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"),
    "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.5",
})
retry = Retry(total=5, backoff_factor=2, status_forcelist=[500, 502, 503, 504],
              allowed_methods=["GET"])
session.mount("https://", HTTPAdapter(max_retries=retry))

In [2]:


class Blocked(Exception):
    pass


def _retry_after(r, default):
    try:
        return max(int(r.headers.get("Retry-After", "")), default)
    except ValueError:
        return default


def fetch(url, params=None, stream=False, timeout=60):
    for attempt in range(4):
        time.sleep(random.uniform(PAUSE_MIN, PAUSE_MAX))
        r = session.get(url, params=params, timeout=timeout, verify=VERIFY_SSL, stream=stream)
        if r.status_code in (403, 429):
            wait = _retry_after(r, 120 * (attempt + 1))
            print(f"  ЕИС ответил {r.status_code}, жду {wait} с (попытка {attempt + 1}/4)")
            r.close()
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r
    raise Blocked("ЕИС отвечает 403/429 - доступ ограничен. Подождите несколько часов и запустите снова.")


def fix_mojibake(s):
    try:
        return s.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s


def filename_from_response(r):
    cd = r.headers.get("Content-Disposition", "")
    m = re.search(r"filename\*\s*=\s*(?:UTF-8|utf-8)''([^;]+)", cd)
    if m:
        return unquote(m.group(1)).strip()
    m = re.search(r'filename\s*=\s*"?([^";]+)"?', cd)
    if m:
        return fix_mojibake(unquote(m.group(1))).strip()
    return None


def safe_name(name, limit=150):
    name = re.sub(r'[\\/:*?"<>|\r\n\t]+', "_", name).strip(" .")
    return name[:limit] or "document"


CTYPE_EXT = {
    "application/pdf": ".pdf",
    "application/msword": ".doc",
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document": ".docx",
    "application/vnd.ms-excel": ".xls",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet": ".xlsx",
    "application/zip": ".zip",
    "application/x-zip-compressed": ".zip",
    "application/rtf": ".rtf",
    "text/plain": ".txt",
}


def guess_ext(ctype):
    return CTYPE_EXT.get(ctype.split(";")[0].strip(), "")


def save_debug(name, html):
    (DEBUG_DIR / name).write_text(html, encoding="utf-8")

In [3]:

COMMON_RE = re.compile(r"common-info\.html\?(?:regNumber|noticeInfoId)=", re.I)


def purchase_key(url):
    q = parse_qs(urlparse(url).query)
    return (q.get("regNumber") or q.get("noticeInfoId") or [url])[0]


def parse_search_page(html):
    soup = BeautifulSoup(html, "lxml")
    raw_keys, items, samples = [], [], []
    seen = set()
    for a in soup.find_all("a", href=COMMON_RE):
        url = urljoin(BASE_URL, a["href"])
        key = purchase_key(url)
        if key in seen:
            continue
        seen.add(key)
        raw_keys.append(key)

        card = a.find_parent(class_="search-registry-entry-block")
        card_text = card.get_text(" ", strip=True) if card else ""
        if card_text and len(samples) < 2:
            samples.append(card_text[:300])
        if card and not TRUST_URL_FILTER and not CUSTOMER_RE.search(card_text):
            continue   

        number = key
        title = ""
        if card:
            hdr = card.select_one(".registry-entry__header-mid__number")
            if hdr:
                m = re.search(r"\d{11,}", hdr.get_text(" ", strip=True))
                if m:
                    number = m.group()
            val = card.select_one(".registry-entry__body-value")
            if val:
                title = val.get_text(" ", strip=True)

        items.append({
            "number": number,
            "key": key,
            "law": "223-ФЗ" if re.search(r"/223/|notice223", url) else "44-ФЗ",
            "customer": "Московский физико-технический институт",
            "title": title,
            "url": url,
            "customer_checked": bool(card),
            "customer_in_text": bool(CUSTOMER_RE.search(card_text)),
        })
    return raw_keys, items, samples


def search_all(search_string):
    purchases, seen_raw = {}, set()
    empty_pages = 0
    for page in range(1, MAX_PAGES + 1):
        params = {
            "searchString": search_string,
            "morphology": "on",
            "search-filter": "Дате размещения",
            "pageNumber": page,
            "sortDirection": "false",
            "recordsPerPage": f"_{PAGE_SIZE}",
            "showLotsInfoHidden": "false",
            "sortBy": "UPDATE_DATE",
            "currencyIdGeneral": "-1",
            "af": "on", "ca": "on", "pc": "on", "pa": "on",  
        }
        if USE_44FZ:
            params["fz44"] = "on"
        if USE_223FZ:
            params["fz223"] = "on"

        if SEARCH_URL_FROM_BROWSER:
            params = parse_qs(urlparse(SEARCH_URL_FROM_BROWSER).query, keep_blank_values=True)
            params["searchString"] = ""      
            params["pageNumber"] = str(page)
            params["recordsPerPage"] = f"_{PAGE_SIZE}"

        r = fetch(SEARCH_URL, params=params)
        if page == 1:
            s0 = BeautifulSoup(r.text, "lxml")
            tot = s0.select_one(".search-results__total")
            inp = s0.select_one("#searchString")
            print("  найдено в ЕИС:", tot.get_text(" ", strip=True) if tot else "?",
                  "| строка поиска на странице:", inp.get("value") if inp else "?")
        raw_keys, items, samples = parse_search_page(r.text)

        if not raw_keys:
            if page == 1:
                save_debug("search_page1.html", r.text)
                print("На первой странице нет карточек. HTML сохранён в", DEBUG_DIR / "search_page1.html",
                      "- возможно, блокировка/капча или изменилась вёрстка.")
            break
        if all(k in seen_raw for k in raw_keys):
            break 
        seen_raw.update(raw_keys)

        for it in items:
            purchases.setdefault(it["key"], it)
        print(f"стр. {page}: карточек {len(raw_keys)}, МФТИ {len(items)}, всего МФТИ {len(purchases)}, заказчик МФТИ виден в тексте карточки у {sum(i['customer_in_text'] for i in items)}")

        if not items:
            empty_pages += 1
            if empty_pages == 1 and not purchases:
                save_debug(f"search_page{page}.html", r.text)
                print("  пример карточек (есть ли в них МФТИ?):")
                for sm in samples:
                    print("   ", sm)
            if empty_pages >= MAX_EMPTY_PAGES:
                print(f"  {MAX_EMPTY_PAGES} страницы подряд без МФТИ - запрос {search_string!r} не работает как фильтр")
                break
        else:
            empty_pages = 0

        if MAX_PURCHASES and len(purchases) >= MAX_PURCHASES:
            break
    result = list(purchases.values())
    return result[:MAX_PURCHASES] if MAX_PURCHASES else result

In [4]:

DOWNLOAD_RE = re.compile(r"filestore|/download/|download\.html", re.I)


def documents_url(card_url):
    return card_url.replace("common-info.html", "documents.html")


def list_documents(card_url, key):
    card = fetch(card_url)
    soup = BeautifulSoup(card.text, "lxml")
    pages = []
    for a in soup.find_all("a", href=True):
        if re.search(r"documents\.html", a["href"], re.I):
            u = urljoin(card.url, a["href"])
            if u not in pages:
                pages.append(u)
    for u in (documents_url(card.url), documents_url(card_url)):
        if u not in pages:
            pages.append(u)

    found, last_err, last_html, loaded = {}, None, "", False
    for page_url in pages[:5]:
        try:
            r = fetch(page_url)
        except requests.HTTPError as e:
            last_err = e
            continue
        loaded = True
        last_html = r.text
        soup = BeautifulSoup(r.text, "lxml")
        for a in soup.find_all("a", href=DOWNLOAD_RE):
            url = urljoin(BASE_URL, a["href"])
            name = (a.get_text(" ", strip=True) or a.get("title") or "").strip()
            if url not in found or len(name) > len(found[url]):
                found[url] = name
        if found:
            break

    if not found:
        save_debug(f"{key}_card.html", card.text)
        if loaded:
            save_debug(f"{key}_documents.html", last_html)
        elif last_err:
            raise last_err
    return [{"name": n, "url": u} for u, n in found.items()]


def download(doc, folder, idx):
    folder.mkdir(parents=True, exist_ok=True)
    r = fetch(doc["url"], stream=True, timeout=180)
    ctype = r.headers.get("Content-Type", "").lower()
    if "text/html" in ctype:
        raise RuntimeError("вместо файла пришла HTML-страница (капча/блокировка?)")
    fname = filename_from_response(r) or doc["name"] or f"document_{idx}"
    fname = safe_name(fname)
    if not Path(fname).suffix:
        fname += guess_ext(ctype)
    path = folder / f"{idx:02d}_{fname}"
    with open(path, "wb") as f:
        for chunk in r.iter_content(65536):
            f.write(chunk)
    return path


def maybe_unpack(path):
    if not (UNPACK_ZIP and path.suffix.lower() == ".zip"):
        return
    target = path.with_suffix("")
    target.mkdir(exist_ok=True)
    try:
        with zipfile.ZipFile(path) as z:
            for info in z.infolist():
                if info.is_dir():
                    continue
                name = info.filename
                if not info.flag_bits & 0x800:    
                    try:
                        name = name.encode("cp437").decode("cp866")
                    except Exception:
                        pass
                out = target / safe_name(Path(name.replace("\\", "/")).name)
                if out.exists():
                    out = target / f"{info.CRC:08x}_{out.name}"
                out.write_bytes(z.read(info))
    except zipfile.BadZipFile:
        print("Битый zip:", path)

In [5]:


def process_purchase(p):
    base = {"purchase": p["number"], "law": p["law"], "customer": p["customer"],
            "title": p["title"], "card_url": p["url"], "filter_sig": FILTER_SIG}
    try:
        docs = list_documents(p["url"], p["key"])
    except Blocked:
        raise
    except Exception as e:
        return [{**base, "status": "error", "error": f"страница документов: {e}"}]
    if not docs:
        return [{**base, "status": "no_documents"}]

    folder = RAW_DIR / safe_name(p["number"])
    rows = []
    for i, d in enumerate(docs, 1):
        is_tz = is_tz_doc(d["name"])
        row = {**base, "doc_name": d["name"], "doc_url": d["url"], "is_tz": is_tz}
        if DOWNLOAD_ONLY_TZ and not is_tz:
            row["status"] = "skipped"
        else:
            try:
                path = download(d, folder, i)
                maybe_unpack(path)
                row.update(status="downloaded", path=str(path))
            except Blocked:
                raise
            except Exception as e:
                row.update(status="error", error=str(e))
        rows.append(row)
    return rows


def load_index():
    
    if not INDEX_FILE.exists():
        return []
    by_purchase = {}
    for line in INDEX_FILE.read_text(encoding="utf-8").splitlines():
        if line.strip():
            row = json.loads(line)
            by_purchase.setdefault(row["purchase"], []).append(row)
    good = [r for rows in by_purchase.values()
            if all(x["status"] != "error" and x.get("filter_sig") == FILTER_SIG for x in rows) for r in rows]
    with open(INDEX_FILE, "w", encoding="utf-8") as f:
        for r in good:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return good


def append_index(rows):
    with open(INDEX_FILE, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [6]:

print("Поиск закупок МФТИ...")
purchases = []
for ss in ([None] if SEARCH_URL_FROM_BROWSER else SEARCH_STRINGS):
    print("Запрос:", ss or "URL из браузера")
    purchases = search_all(ss)
    if purchases:
        break
print("К обработке закупок:", len(purchases))

all_rows = load_index()
done = {r["purchase"] for r in all_rows}
print("Уже обработано ранее:", len(done))

todo = [p for p in purchases if p["number"] not in done]
if MAX_PER_RUN:
    todo = todo[:MAX_PER_RUN]
print("Новых закупок в этом запуске:", len(todo))

fails = 0
try:
    for n, p in enumerate(tqdm(todo), 1):
        rows = process_purchase(p)
        append_index(rows)
        all_rows.extend(rows)

        fails = fails + 1 if all(r["status"] == "error" for r in rows) else 0
        if fails >= MAX_CONSECUTIVE_FAILURES:
            print(f"\n{fails} закупок подряд с ошибками - останавливаюсь, чтобы не нагружать ЕИС. "
                  "Посмотрите ошибки в итоговой ячейке и запустите позже.")
            break
        if n % LONG_PAUSE_EVERY == 0:
            time.sleep(random.uniform(*LONG_PAUSE))
except Blocked as e:
    print("\nОСТАНОВКА:", e)
except KeyboardInterrupt:
    print("\nПрервано вручную, прогресс сохранён.")

print("Готово")

Поиск закупок МФТИ...
Запрос: URL из браузера
  найдено в ЕИС: более 4 500 записей | строка поиска на странице: None
стр. 1: карточек 50, МФТИ 50, всего МФТИ 50, заказчик МФТИ виден в тексте карточки у 50
стр. 2: карточек 50, МФТИ 50, всего МФТИ 100, заказчик МФТИ виден в тексте карточки у 50
стр. 3: карточек 50, МФТИ 50, всего МФТИ 150, заказчик МФТИ виден в тексте карточки у 50
стр. 4: карточек 50, МФТИ 50, всего МФТИ 200, заказчик МФТИ виден в тексте карточки у 50
стр. 19: карточек 50, МФТИ 50, всего МФТИ 950, заказчик МФТИ виден в тексте карточки у 50
стр. 20: карточек 50, МФТИ 50, всего МФТИ 1000, заказчик МФТИ виден в тексте карточки у 50
стр. 21: карточек 50, МФТИ 50, всего МФТИ 1050, заказчик МФТИ виден в тексте карточки у 50
стр. 22: карточек 50, МФТИ 50, всего МФТИ 1100, заказчик МФТИ виден в тексте карточки у 50
стр. 23: карточек 50, МФТИ 50, всего МФТИ 1150, заказчик МФТИ виден в тексте карточки у 50
стр. 24: карточек 50, МФТИ 50, всего МФТИ 1200, заказчик МФТИ виден в текс

  0%|          | 0/167 [00:00<?, ?it/s]


5 закупок подряд с ошибками - останавливаюсь, чтобы не нагружать ЕИС. Посмотрите ошибки в итоговой ячейке и запустите позже.
Готово


In [7]:

df = pd.DataFrame(all_rows)
if df.empty:
    print("Пусто. Смотрите", DEBUG_DIR)
else:
    df.to_csv(OUT_DIR / "documents_index.csv", index=False, encoding="utf-8-sig")
    print("Закупок:", df["purchase"].nunique())
    print(df["status"].value_counts().to_string())
    if (df["status"] == "no_documents").mean() > 0.5:
        print("\nУ большинства закупок не найдены документы - откройте файлы в", DEBUG_DIR,
              "и проверьте вёрстку вкладки «Документы».")
    err = df[df["status"] == "error"]
    if len(err):
        print("\nОшибки (повторный запуск ячейки запуска попробует их ещё раз):")
        pd.set_option("display.max_colwidth", 300)
        cols = [c for c in ["purchase", "card_url", "doc_name", "error"] if c in err.columns]
        print(err[cols].head(10).to_string())
df.head(20)

Закупок: 4463
status
skipped       6229
downloaded    5312
error           34

Ошибки (повторный запуск ячейки запуска попробует их ещё раз):
          purchase                                                                                         card_url                                   doc_name                                                                                                                                                                                                       error
11367  31807389954  https://zakupki.gov.ru/223/purchase/public/purchase/info/common-info.html?regNumber=31807389954           Изменения в документацию К18-022  HTTPSConnectionPool(host='zakupki.gov.ru', port=443): Max retries exceeded with url: /223/filestore/public/1.0/download/fz223/file.html?uid=null (Caused by ResponseError('too many 500 error responses'))
11383  31807012773  https://zakupki.gov.ru/223/purchase/public/purchase/info/common-info.html?regNumber=31807012773                 

,purchase,law,customer,title,card_url,filter_sig,doc_name,doc_url,is_tz,status,path,error
0,32211220037,223-ФЗ,Московский физико-технический институт,"Выполнение работ по текущему ремонту первого этажа учебно – производственных мастерских МФТИ, Физтех",https://zakupki.gov.ru/223/purchase/public/purchase/info/common-info.html?regNumber=32211220037,4cae4838,Документация К22-002.doc,https://zakupki.gov.ru/223/filestore/public/1.0/download/fz223/file.html?uid=12570644D8FF7704E063D4548D0A4E16,True,downloaded,mipt_tz_dataset/raw/32211220037/01_Документация К22-002.doc,NaN
1,32211220037,223-ФЗ,Московский физико-технический институт,"Выполнение работ по текущему ремонту первого этажа учебно – производственных мастерских МФТИ, Физтех",https://zakupki.gov.ru/223/purchase/public/purchase/info/common-info.html?regNumber=32211220037,4cae4838,Разъяснение 1.pdf,https://zakupki.gov.ru/223/filestore/public/1.0/download/fz223/file.html?uid=129CD561FCF71651E063D4548D0AF020,False,skipped,NaN,NaN
2,32211220037,223-ФЗ,Московский физико-технический институт,"Выполнение работ по текущему ремонту первого этажа учебно – производственных мастерских МФТИ, Физтех",https://zakupki.gov.ru/223/purchase/public/purchase/info/common-info.html?regNumber=32211220037,4cae4838,Протокол рассмотрения первых частей заявок К22-002 от 01.04.2022.pdf,https://zakupki.gov.ru/223/filestore/public/1.0/download/fz223/file.html?uid=1257062DC5F06B31E063D4548D0A522E,False,skipped,NaN,NaN
3,32211220037,223-ФЗ,Московский физико-технический институт,"Выполнение работ по текущему ремонту первого этажа учебно – производственных мастерских МФТИ, Физтех",https://zakupki.gov.ru/223/purchase/public/purchase/info/common-info.html?regNumber=32211220037,4cae4838,Итоговый протокол №К22-002 от 05.04.2022.pdf,https://zakupki.gov.ru/223/filestore/public/1.0/download/fz223/file.html?uid=12CF6BEF6BDB1327E063D4548D0AAB88,False,skipped,NaN,NaN
4,32211220037,223-ФЗ,Московский физико-технический институт,"Выполнение работ по текущему ремонту первого этажа учебно – производственных мастерских МФТИ, Физтех",https://zakupki.gov.ru/223/purchase/public/purchase/info/common-info.html?regNumber=32211220037,4cae4838,Итоговый протокол №К22-002 от 05.04.2022.pdf,https://zakupki.gov.ru/223/filestore/public/1.0/download/fz223/file.html?uid=11CB5E96AB901ECDE063D4548D0A2994,False,skipped,NaN,NaN
5,32211214035,223-ФЗ,Московский физико-технический институт,Оказание услуг по созданию проекта виртуальной реальности,https://zakupki.gov.ru/223/purchase/public/purchase/info/common-info.html?regNumber=32211214035,4cae4838,Извещение,https://zakupki.gov.ru/223/filestore/public/1.0/download/fz223/file.html?uid=1301B4A530417111E063D4548D0A2AE5,True,downloaded,mipt_tz_dataset/raw/32211214035/01_Извещение.pdf,NaN
6,32211214035,223-ФЗ,Московский физико-технический институт,Оказание услуг по созданию проекта виртуальной реальности,https://zakupki.gov.ru/223/purchase/public/purchase/info/common-info.html?regNumber=32211214035,4cae4838,Протокол,https://zakupki.gov.ru/223/filestore/public/1.0/download/fz223/file.html?uid=11CB5EA6B7181ED1E063D4548D0A65C3,False,skipped,NaN,NaN
7,32211198663,223-ФЗ,Московский физико-технический институт,Поставка боксов микробиологической безопасности,https://zakupki.gov.ru/223/purchase/public/purchase/info/common-info.html?regNumber=32211198663,4cae4838,Извещение З22-007.doc,https://zakupki.gov.ru/223/filestore/public/1.0/download/fz223/file.html?uid=11CB5E797B3C1DBEE063D4548D0A8C94,True,downloaded,mipt_tz_dataset/raw/32211198663/01_Извещение З22-007.doc,NaN
8,32211198663,223-ФЗ,Московский физико-технический институт,Поставка боксов микробиологической безопасности,https://zakupki.gov.ru/223/purchase/public/purchase/info/common-info.html?regNumber=32211198663,4cae4838,Итоговый протокол №З22-007 от 15.03.2022.pdf,https://zakupki.gov.ru/223/filestore/public/1.0/download/fz223/file.html?uid=11CB5E80DE471ED1E063D4548D0A65C3,False,skipped,NaN,NaN
9,32211196649,223-ФЗ,Московский физико-технический институт,По